# Cross-technology gene UMI count comparison

Build a gene × dataset matrix of **total UMI counts per gene** across the 5 benchmark technologies, then cluster to identify gene groups with different coverage patterns across technologies.

**Input**: `inference_mudata.h5mu` for each of the 5 benchmark datasets (`Benchmark_basic_run_threshold_cleanser` for 4 + `Benchmark_basic_run_threshold` sceptre substitute for Gersbach HTv2, which has no cleanser h5mu).

**Output** (`results/cross_tech_comparison/cross_tech_umi_counts/`):
- `gene_total_umi_matrix.tsv.gz` — gene × dataset raw UMI counts (0 for missing genes)
- `gene_total_umi_clustermap.pdf` — clustered heatmap (log1p, row-normalized)
- `gene_cluster_membership.tsv` — per-gene cluster assignment from row dendrogram

To rerun with different runs/datasets, edit the manifest at the top.


In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import mudata as mu
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist

REPO = '/carter/users/aklie/projects/tf_perturb_seq'
MANIFEST = f'{REPO}/CRISPRi_tech_benchmark/manifests/basic_threshold_mudata_paths_local.tsv'
OUTDIR = f'{REPO}/CRISPRi_tech_benchmark/results/cross_tech_comparison/cross_tech_umi_counts'
os.makedirs(OUTDIR, exist_ok=True)

manifest = pd.read_csv(MANIFEST, sep='\t')
manifest

## 1. Extract per-gene total UMI counts from each dataset

Load the gene modality from each `inference_mudata.h5mu`, sum the raw count matrix across cells, and record per-gene totals indexed by the gene's `var.index` (ENSEMBL ID from the IGVF reference). No normalization yet — we want raw absolute UMI counts as Adam specified.

In [ ]:
totals = {}  # dataset short_name -> pd.Series indexed by gene id
for _, row in manifest.iterrows():
    short, path = row['short_name'], row['mudata_path']
    print(f'Loading {short} ({path.split("/")[-4]})...')
    mdata = mu.read_h5mu(path, backed='r')
    gene_ad = mdata.mod['gene']
    X = gene_ad.X
    # Sum across cells (axis=0) -> per-gene total UMI count
    if sparse.issparse(X):
        gene_totals = np.asarray(X.sum(axis=0)).flatten()
    else:
        gene_totals = X.sum(axis=0)
    totals[short] = pd.Series(gene_totals.astype(np.int64), index=gene_ad.var.index.astype(str), name=short)
    print(f'  n_cells={gene_ad.n_obs:>7d}  n_genes={gene_ad.n_vars:>6d}  total_umi={gene_totals.sum():,.0f}')
    mdata.file.close()
print('Done.')

## 2. Build the gene × dataset matrix

Union of all gene IDs across the 5 datasets; missing genes filled with 0 (per the agreed convention).

In [ ]:
mat = pd.concat(totals.values(), axis=1).fillna(0).astype(np.int64)
mat.columns = list(totals.keys())
print(f'gene × dataset matrix: {mat.shape[0]:,} genes × {mat.shape[1]} datasets')
print(f'  Genes detected in all {mat.shape[1]} datasets: {(mat > 0).all(axis=1).sum():,}')
print(f'  Genes detected in only 1 dataset:           {((mat > 0).sum(axis=1) == 1).sum():,}')
print(f'  Genes with 0 UMI everywhere:                {(mat.sum(axis=1) == 0).sum():,}')
mat.head()

In [ ]:
# Drop genes with 0 counts in all datasets (they exist only in the IGVF reference but never detected)
mat_nonzero = mat.loc[mat.sum(axis=1) > 0].copy()
print(f'After dropping all-zero genes: {mat_nonzero.shape[0]:,} genes')

# Save the raw count matrix
out_tsv = f'{OUTDIR}/gene_total_umi_matrix.tsv.gz'
mat_nonzero.to_csv(out_tsv, sep='\t', compression='gzip')
print(f'Wrote {out_tsv}')

## 3. Clustered heatmap

log1p-transform the raw counts to make the dynamic range plottable, then row-normalize (z-score across datasets) so the heatmap shows each gene's *pattern* across datasets rather than its absolute count level (otherwise high-expression genes would dominate). Hierarchical clustering on rows (genes) and columns (datasets).

In [ ]:
# log1p
lmat = np.log1p(mat_nonzero)

# Row z-score (so each gene's pattern is shown, not its absolute level)
zmat = lmat.sub(lmat.mean(axis=1), axis=0).div(lmat.std(axis=1).replace(0, 1), axis=0)

# Clustermap
g = sns.clustermap(
    zmat,
    cmap='RdBu_r',
    vmin=-2, vmax=2, center=0,
    row_cluster=True, col_cluster=True,
    yticklabels=False,
    figsize=(6, 10),
    cbar_kws={'label': 'log1p(UMI), row z-score'},
    method='ward',
)
g.ax_heatmap.set_xlabel('Dataset')
g.ax_heatmap.set_ylabel(f'Gene (n={zmat.shape[0]:,})')
plt.suptitle('Cross-technology total gene UMI counts\n(row-normalized log1p)', y=1.02)
plt.savefig(f'{OUTDIR}/gene_total_umi_clustermap.pdf', bbox_inches='tight')
plt.show()
print(f'Wrote {OUTDIR}/gene_total_umi_clustermap.pdf')

## 4. Identify gene clusters from the row dendrogram

Cut the row linkage at K=8 clusters (tune as needed); each cluster groups genes with a similar across-technology pattern. Per-cluster mean profile shown as a small panel.

In [ ]:
K = 8  # number of gene clusters to call from the row dendrogram
row_linkage = linkage(pdist(zmat.values, metric='euclidean'), method='ward')
cluster_ids = fcluster(row_linkage, t=K, criterion='maxclust')

clusters = pd.DataFrame({'gene_id': zmat.index, 'cluster': cluster_ids})
cluster_sizes = clusters['cluster'].value_counts().sort_index()
print('Cluster sizes:')
print(cluster_sizes)

out_clust = f'{OUTDIR}/gene_cluster_membership.tsv'
clusters.to_csv(out_clust, sep='\t', index=False)
print(f'Wrote {out_clust}')

In [ ]:
# Per-cluster mean profile across datasets
profile = zmat.groupby(cluster_ids).mean()
profile.index = [f'C{c} (n={cluster_sizes[c]:,})' for c in profile.index]

fig, ax = plt.subplots(figsize=(6, 0.4 * K + 1))
sns.heatmap(profile, cmap='RdBu_r', vmin=-2, vmax=2, center=0,
            annot=True, fmt='.2f', cbar_kws={'label': 'mean z-score'}, ax=ax)
ax.set_xlabel('Dataset')
ax.set_ylabel('Cluster')
ax.set_title(f'Per-cluster mean pattern across technologies (K={K})')
plt.tight_layout()
plt.savefig(f'{OUTDIR}/gene_cluster_mean_profile.pdf', bbox_inches='tight')
plt.show()
print(f'Wrote {OUTDIR}/gene_cluster_mean_profile.pdf')

## 5. Quick top-gene examples per cluster

For each cluster, show the 5 genes with the highest absolute total UMI count (across all 5 datasets) — quick gut-check of what kind of genes each cluster contains.

In [ ]:
raw_totals = mat_nonzero.sum(axis=1).rename('total_umi_all_datasets')
ann = clusters.merge(raw_totals.reset_index().rename(columns={'index': 'gene_id'}), on='gene_id')
top_per_cluster = (ann.sort_values(['cluster', 'total_umi_all_datasets'], ascending=[True, False])
                      .groupby('cluster').head(5))
print(top_per_cluster.to_string(index=False))